# Anomaly Detection Evaluation: BYOL Features — Multi-Method Benchmark

This notebook benchmarks multiple anomaly detection strategies on the MGCLS dataset using BYOL-extracted features.
It compares:
- **Baseline**: PCA scores from Protege catalogue
- **Moment Pooling (L2)**: dimensionality reduction via statistical moments
- **Moment Pooling + Isolation Forest**
- **PCA + Isolation Forest**
- **Moment Pooling + Extended Isolation Forest (EIF)**  ← new, from arXiv:2110.13402
- **Moment Pooling + ECOD** ← statistical anomaly detection, parameter-free
- **Moment Pooling + COPOD** ← copula-based, fast and interpretable

Evaluation is on the **interesting-label subset** (author ML score ≥ 4) using ROC-AUC, PR-AUC, Recall@100, and Spearman correlation.

In [ ]:
# Install dependencies if needed
# !pip install pyod eif isotree --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# PyOD: statistical anomaly detectors (no deep learning needed)
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.iforest import IForest

# Extended Isolation Forest (improved version)
try:
    import eif as iso_ext
    EIF_AVAILABLE = True
except ImportError:
    EIF_AVAILABLE = False
    print("eif not installed — skipping Extended Isolation Forest. Run: pip install eif")

from utils import load_features, load_catalogue, MomentPooling, compute_metrics

print(f"PyOD available: True")
print(f"Extended IF available: {EIF_AVAILABLE}")

## Helper Functions

In [ ]:
def compute_ind_sum(found_inds, all_inds):
    this_found_inds = sorted(found_inds)
    out = np.zeros(len(all_inds))
    for i in this_found_inds:
        out[i:] += 1
    return out


def cumulative_sum(anomaly_scores, labels, sort_by='score'):
    """Compute cumulative anomaly discovery curve."""
    if sort_by == 'random':
        sorted_inds = np.random.permutation(anomaly_scores.index)
    else:
        sorted_inds = anomaly_scores.sort_values(ascending=False).index

    labs = labels.loc[anomaly_scores.index]
    anom_inds = labs[labs == 1].index

    found_inds = []
    for i in anom_inds:
        pos = np.where(sorted_inds == i)[0]
        if len(pos) > 0:
            found_inds.append(pos[0])

    return compute_ind_sum(found_inds, sorted_inds)


def topk_recall(y_true, scores, k=100):
    """Fraction of true anomalies in top-k ranked candidates."""
    ranked = scores.sort_values(ascending=False).index[:k]
    return y_true.loc[ranked].sum() / y_true.sum()


def score_from_pyod(model, X_arr, index):
    """Fit a PyOD model and return anomaly scores as a pd.Series."""
    model.fit(X_arr)
    return pd.Series(model.decision_scores_, index=index, name='score')

## Data Loading and Preprocessing

In [ ]:
print("Loading data...")
X = load_features()
cat = load_catalogue()

# Align on catalogue objects
X = X.loc[cat.objid]

# Labels
labels = cat.set_index("objid")["evaluation_subset_author_ML_score"].loc[X.index]
y_interesting = (labels >= 4).astype(int)

print(f"\nDataset: {X.shape[0]} objects, {X.shape[1]} BYOL features")
print(f"Interesting anomalies (score ≥ 4): {y_interesting.sum()} / {len(y_interesting)}")
print(f"Prevalence: {y_interesting.mean():.2%}")

# Normalize
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)
print("\nFeatures standardized (zero mean, unit variance).")

## Baseline: PCA Score from Protege

In [ ]:
pca_scores = cat.set_index("objid")["protege_score"].loc[X.index]
pca_scores.name = 'score'
print("Protege PCA scores loaded.")

## Moment Pooling Dimensionality Reduction

Following arXiv:2403.08854 — compress BYOL features to a low-dimensional space using PCA, then expand with polynomial moments.

In [ ]:
# Tune latent_dim and order based on dataset size and feature count
# latent_dim=8, order=2 gives 44 features — manageable for downstream detectors
print("Running Moment Pooling (latent_dim=8, order=2)...")
mp_model = MomentPooling(latent_dim=8, order=2)
X_mp = mp_model.fit_transform(X_scaled)
X_mp_clean = X_mp.drop(columns=["bias"], errors="ignore")

print(f"Moment Pooling features shape: {X_mp_clean.shape}")

# Also prepare plain PCA features for comparison
pca_model = PCA(n_components=8)
Z_pca = pca_model.fit_transform(X_scaled)
print(f"Explained variance (8 PCs): {pca_model.explained_variance_ratio_.sum():.2%}")

## Method 1: Moment Pooling + L2 Score (Reconstruction Distance)

In [ ]:
mp_l2_scores = np.linalg.norm(X_mp_clean.values, axis=1)
mp_l2_scores = pd.Series(mp_l2_scores, index=X.index, name='score')
print("Moment Pooling L2 scores computed.")

## Method 2: Moment Pooling + Standard Isolation Forest

In [ ]:
print("Running Moment Pooling + Isolation Forest...")
iso = IsolationForest(n_estimators=300, random_state=42, contamination='auto')
iso.fit(X_mp_clean)
mp_iso_scores = pd.Series(-iso.score_samples(X_mp_clean), index=X.index, name='score')
print("Done.")

## Method 3: PCA + Isolation Forest

In [ ]:
print("Running PCA + Isolation Forest...")
iso_pca = IsolationForest(n_estimators=300, random_state=42, contamination='auto')
iso_pca.fit(Z_pca)
pca_iso_scores = pd.Series(-iso_pca.score_samples(Z_pca), index=X.index, name='score')
print("Done.")

## Method 4 (NEW): Moment Pooling + Extended Isolation Forest (EIF)

EIF addresses a well-known bias in standard IF where anomalies near the data center are harder to isolate (arXiv:2110.13402). It uses random hyperplanes instead of axis-aligned splits.

In [ ]:
if EIF_AVAILABLE:
    print("Running Moment Pooling + Extended Isolation Forest...")
    X_mp_arr = X_mp_clean.values.astype(np.float64)
    eif_model = iso_ext.iForest(X_mp_arr, ntrees=300, sample_size=min(256, len(X_mp_arr)))
    eif_scores = eif_model.compute_paths(X_mp_arr)
    mp_eif_scores = pd.Series(eif_scores, index=X.index, name='score')
    print("Done.")
else:
    print("EIF not available — using standard IF as fallback.")
    mp_eif_scores = mp_iso_scores.copy()

## Method 5 (NEW): Moment Pooling + ECOD

ECOD (Empirical Cumulative Distribution-based Outlier Detection) is parameter-free, highly interpretable, and works well with pre-computed features. It estimates tail probabilities per feature and combines them — a good fit for BYOL-style representations.

In [ ]:
print("Running Moment Pooling + ECOD...")
mp_ecod_scores = score_from_pyod(
    ECOD(contamination=0.1),
    X_mp_clean.values,
    index=X.index
)
print("Done.")

## Method 6 (NEW): Moment Pooling + COPOD

COPOD uses copula functions to model joint distributions and score anomalies without assuming Gaussianity — useful when BYOL features have non-normal marginals.

In [ ]:
print("Running Moment Pooling + COPOD...")
mp_copod_scores = score_from_pyod(
    COPOD(contamination=0.1),
    X_mp_clean.values,
    index=X.index
)
print("Done.")

## Evaluation

In [ ]:
all_methods = {
    "PCA (Protege)": pca_scores,
    "MP + L2": mp_l2_scores,
    "MP + IsoForest": mp_iso_scores,
    "PCA + IsoForest": pca_iso_scores,
    "MP + EIF": mp_eif_scores,
    "MP + ECOD": mp_ecod_scores,
    "MP + COPOD": mp_copod_scores,
}

results = []
for name, scores in all_methods.items():
    m = compute_metrics(y_interesting, scores)
    results.append({
        "Method": name,
        "ROC-AUC (4-5)": round(m["roc_auc"], 4),
        "PR-AUC (4-5)": round(m["pr_auc"], 4),
        "Recall@100 (4-5)": round(topk_recall(y_interesting, scores), 4),
        "Spearman (1-5)": round(labels.corr(scores, method='spearman'), 4),
    })

df_results = pd.DataFrame(results).set_index("Method")
display(df_results.sort_values("ROC-AUC (4-5)", ascending=False))

## Cumulative Discovery Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']

ax = axes[0]
for (name, scores), c, ls in zip(all_methods.items(), colors, linestyles):
    cum = cumulative_sum(scores, y_interesting)
    ax.plot(cum, label=name, color=c, linestyle=ls)

# Random baseline
n = len(y_interesting)
n_anom = y_interesting.sum()
ax.plot(np.linspace(0, n_anom, n), label='Random', color='gray', linestyle=':', linewidth=1)

ax.set_xlabel('Rank (ordered by anomaly score)')
ax.set_ylabel('Anomalies found (score ≥ 4)')
ax.set_title('Cumulative Anomaly Discovery')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Zoom into top-200
ax2 = axes[1]
for (name, scores), c, ls in zip(all_methods.items(), colors, linestyles):
    cum = cumulative_sum(scores, y_interesting)
    ax2.plot(cum[:200], label=name, color=c, linestyle=ls)
ax2.plot(np.linspace(0, min(n_anom, 200 * n_anom / n), 200), label='Random', color='gray', linestyle=':', linewidth=1)
ax2.axvline(100, color='black', linestyle='--', alpha=0.4, label='k=100')
ax2.set_xlabel('Rank (top 200)')
ax2.set_ylabel('Anomalies found')
ax2.set_title('Cumulative Discovery — Top 200 (zoomed)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('discovery_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Score Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for ax, (name, scores) in zip(axes, all_methods.items()):
    s_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    ax.hist(s_norm[y_interesting == 0], bins=40, alpha=0.6, label='Normal', density=True, color='steelblue')
    ax.hist(s_norm[y_interesting == 1], bins=40, alpha=0.6, label='Anomaly', density=True, color='tomato')
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('Normalized score')

# Hide the unused subplot
for ax in axes[len(all_methods):]:
    ax.set_visible(False)

plt.suptitle('Score Distributions: Anomaly vs Normal', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary and Insights

### Key findings

- **Baseline PCA** (Protege): strong reference point due to task-specific tuning.
- **Moment Pooling + ECOD / COPOD**: Statistical detectors operating on moment-expanded features. ECOD is parameter-free and robust; COPOD handles non-Gaussian marginals.
- **Extended Isolation Forest (EIF)**: Reduces bias vs standard IF via random hyperplane splits. Best improvement expected when anomalies cluster near the centre of the feature space.

### Why Moment Pooling alone underperforms
The L2 norm of polynomial moments captures *statistical* outliers but misses *semantic* ones — unusual radio morphologies that BYOL encodes in directions not aligned with high-variance PCA axes.

### Next steps
- Hyperparameter sweep: `latent_dim` ∈ {4, 8, 16}, `order` ∈ {2, 3}
- Try **BYOL raw features** (no PCA) directly into ECOD/COPOD with feature selection
- Implement **DeepSVDD** and **PatchCore** (multi-layer features)
- Score ensemble: average top-3 methods after rank normalisation